# 01 — Data Collection

Pulls NBA team game logs, rosters, and player game logs via `nba_api` for seasons 2020-21 through 2024-25.  
All raw data is saved to `data/raw/` immediately after fetching. Re-running this notebook skips files that already exist.

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import glob

import pandas as pd

from src.scraper import (
    fetch_all_teams,
    fetch_player_stats,
    fetch_team_game_logs,
    fetch_team_roster,
)

SEASONS = ["2020-21", "2021-22", "2022-23", "2023-24", "2024-25"]
TOP_PLAYERS_PER_TEAM = 8  # top players by games played on each roster

## 1. All NBA Teams

In [2]:
teams_df = fetch_all_teams()
print(f"{len(teams_df)} teams loaded")
teams_df.head()

30 teams loaded


,id,full_name,abbreviation,nickname,city,state,year_founded
0,1610612737,Atlanta Hawks,ATL,Hawks,Atlanta,Georgia,1949
1,1610612738,Boston Celtics,BOS,Celtics,Boston,Massachusetts,1946
2,1610612739,Cleveland Cavaliers,CLE,Cavaliers,Cleveland,Ohio,1970
3,1610612740,New Orleans Pelicans,NOP,Pelicans,New Orleans,Louisiana,2002
4,1610612741,Chicago Bulls,CHI,Bulls,Chicago,Illinois,1966


## 2. Team Game Logs

One CSV per team × season → `data/raw/team_game_logs_{team_id}_{season}.csv`

In [3]:
total = len(teams_df) * len(SEASONS)
done = 0

for _, team in teams_df.iterrows():
    for season in SEASONS:
        df = fetch_team_game_logs(team["id"], season)
        done += 1
        if done % 10 == 0 or done == total:
            print(f"  [{done}/{total}] {team['abbreviation']} {season} — {len(df)} games")

print("Done.")

  [10/150] BOS 2024-25 — 82 games
  [20/150] NOP 2024-25 — 82 games
  [30/150] DAL 2024-25 — 82 games
  [40/150] GSW 2024-25 — 82 games
  [50/150] LAC 2024-25 — 82 games
  [60/150] MIA 2024-25 — 82 games
  [70/150] MIN 2024-25 — 82 games
  [80/150] NYK 2024-25 — 82 games
  [90/150] IND 2024-25 — 82 games
  [100/150] PHX 2024-25 — 82 games
  [110/150] SAC 2024-25 — 82 games
  [120/150] OKC 2024-25 — 82 games
  [130/150] UTA 2024-25 — 82 games
  [140/150] WAS 2024-25 — 82 games
  [150/150] CHA 2024-25 — 82 games
Done.


## 3. Team Rosters

One CSV per team × season → `data/raw/roster_{team_id}_{season}.csv`

In [4]:
total = len(teams_df) * len(SEASONS)
done = 0

for _, team in teams_df.iterrows():
    for season in SEASONS:
        df = fetch_team_roster(team["id"], season)
        done += 1
        if done % 20 == 0 or done == total:
            print(f"  [{done}/{total}] {team['abbreviation']} {season} — {len(df)} players")

print("Done.")

  [20/150] NOP 2024-25 — 18 players
  [40/150] GSW 2024-25 — 17 players
  [60/150] MIA 2024-25 — 18 players
  [80/150] NYK 2024-25 — 18 players
  [100/150] PHX 2024-25 — 18 players
  [120/150] OKC 2024-25 — 18 players
  [140/150] WAS 2024-25 — 18 players
  [150/150] CHA 2024-25 — 18 players
Done.


## 4. Player Game Logs

For each team roster, take the top `TOP_PLAYERS_PER_TEAM` players by games played and fetch their game-level stats.  
One CSV per player × season → `data/raw/player_game_logs_{player_id}_{season}.csv`

In [5]:
fetched_pairs = set()  # (player_id, season) already pulled
total_fetched = 0

for _, team in teams_df.iterrows():
    for season in SEASONS:
        roster = fetch_team_roster(team["id"], season)

        # roster has a GP (games played) column; fall back to first N rows if missing
        if "GP" in roster.columns:
            top_players = roster.nlargest(TOP_PLAYERS_PER_TEAM, "GP")
        else:
            top_players = roster.head(TOP_PLAYERS_PER_TEAM)

        for _, player in top_players.iterrows():
            pid = int(player["PLAYER_ID"])
            key = (pid, season)
            if key in fetched_pairs:
                continue  # player appeared on multiple team rosters same season
            fetched_pairs.add(key)
            fetch_player_stats(pid, season)
            total_fetched += 1
            if total_fetched % 50 == 0:
                print(f"  {total_fetched} player-seasons fetched so far...")

print(f"Done. {total_fetched} unique player-season files.")

  50 player-seasons fetched so far...
  100 player-seasons fetched so far...
  150 player-seasons fetched so far...
  200 player-seasons fetched so far...
  250 player-seasons fetched so far...
  300 player-seasons fetched so far...
  350 player-seasons fetched so far...
  400 player-seasons fetched so far...
  450 player-seasons fetched so far...
  500 player-seasons fetched so far...
  550 player-seasons fetched so far...
  600 player-seasons fetched so far...
  650 player-seasons fetched so far...
  700 player-seasons fetched so far...
  750 player-seasons fetched so far...
  800 player-seasons fetched so far...
  850 player-seasons fetched so far...
  900 player-seasons fetched so far...
  950 player-seasons fetched so far...
  1000 player-seasons fetched so far...
  1050 player-seasons fetched so far...
  1100 player-seasons fetched so far...
  1150 player-seasons fetched so far...
  1200 player-seasons fetched so far...
Done. 1200 unique player-season files.


## 5. Summary

In [ ]:
raw_dir = Path.cwd().parent / "data" / "raw"

n_gamelogs = len(list(raw_dir.glob("team_game_logs_*.csv")))
n_rosters  = len(list(raw_dir.glob("roster_*.csv")))
n_players  = len(list(raw_dir.glob("player_game_logs_*.csv")))

print(f"Team game log files : {n_gamelogs}  (expect {len(teams_df) * len(SEASONS)} = 30×5)")
print(f"Roster files        : {n_rosters}  (expect {len(teams_df) * len(SEASONS)} = 30×5)")
print(f"Player log files    : {n_players}")

# quick row-count check on one team
sample = list(raw_dir.glob("team_game_logs_*.csv"))[0]
sample_df = pd.read_csv(sample)
print(f"\nSample file: {sample.name}")
print(f"  Rows: {len(sample_df)}, Columns: {list(sample_df.columns)}")

Team game log files : 150  (expect 150 = 30×5)
Roster files        : 150  (expect 150 = 30×5)
Player log files    : 1200

Sample file: team_game_logs_1610612751_2022-23.csv
  Rows: 82, Columns: ['Team_ID', 'Game_ID', 'GAME_DATE', 'MATCHUP', 'WL', 'W', 'L', 'W_PCT', 'MIN', 'FGM', 'FGA', 'FG_PCT', 'FG3M', 'FG3A', 'FG3_PCT', 'FTM', 'FTA', 'FT_PCT', 'OREB', 'DREB', 'REB', 'AST', 'STL', 'BLK', 'TOV', 'PF', 'PTS']


,Team_ID,Game_ID,GAME_DATE,MATCHUP,WL,W,L,W_PCT,MIN,FGM,...,FT_PCT,OREB,DREB,REB,AST,STL,BLK,TOV,PF,PTS
0,1610612751,22201217,"APR 09, 2023",BKN vs. PHI,L,45,37,0.549,240,35,...,0.821,10,32,42,22,6,6,18,22,105
1,1610612751,22201207,"APR 07, 2023",BKN vs. ORL,W,45,36,0.556,240,35,...,0.880,10,35,45,27,14,3,9,12,101
2,1610612751,22201189,"APR 05, 2023",BKN @ DET,W,44,36,0.550,240,45,...,0.696,11,27,38,36,6,7,11,12,123
3,1610612751,22201180,"APR 04, 2023",BKN vs. MIN,L,43,36,0.544,240,39,...,0.786,7,33,40,21,6,8,9,18,102
4,1610612751,22201164,"APR 02, 2023",BKN vs. UTA,W,43,35,0.551,240,38,...,0.765,7,33,40,25,6,2,8,26,111
